# NevoScan
## Эксперимент 7: 4ch vs 5ch (маска невуса как 5-й канал)

**Гипотеза:** модель в эксп. 6 иногда предсказывает признаки, опираясь не на саму родинку, а на окружающий фон. Если добавить маску невуса как 5-й канал входа, модель явно получит сигнал "здесь родинка", что должно улучшить качество предсказания и сделать Grad-CAM более сфокусированным на области очага.

**Дизайн эксперимента:**
- Архитектура — `MultiHeadEfficientNet` из эксп.6, 8 голов
- Вход для baseline: 4 канала (RGB + маска признаков), как в эксп.6
- Вход для нашего варианта: 5 каналов (RGB + маска признаков + маска невуса из BiRefNet)
- Все остальные гиперпараметры (batch, lr, epochs, focal α/γ, fn_weight, head_weights, augmentations, splits) идентичны
- Один и тот же random_seed=42, чтобы сравнение было честным
- Сравнительные Grad-CAM на одних и тех же изображениях

**Маски невуса:** предрасчитаны в `precompute_lesion_masks.ipynb` (схема A2 — BiRefNet без детектора), лежат в `/content/drive/MyDrive/Диплом/lesion_masks/`.

In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

### 1. Монтируем Drive и распаковываем данные

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Derm7pt
os.makedirs('/content/dataset', exist_ok=True)
if not os.path.exists('/content/dataset/release_v0'):
    os.system('unzip -q "/content/drive/MyDrive/Диплом/практика_преддипломная/derm7pt.zip" '
              '-d "/content/dataset"')
    print('✓ Derm7pt распакован')
else:
    print('✓ Derm7pt уже есть')

# ISIC изображения
os.makedirs('/content/isic', exist_ok=True)
if not os.path.exists('/content/isic/images'):
    os.system('wget -q --show-progress -O /content/isic/images.zip '
              'https://isic-challenge-data.s3.amazonaws.com/2018/'
              'ISIC2018_Task1-2_Training_Input.zip')
    os.system('unzip -q /content/isic/images.zip -d /content/isic/tmp')
    os.system('mv "/content/isic/tmp/ISIC2018_Task1-2_Training_Input" /content/isic/images')
    os.system('rm -rf /content/isic/images.zip /content/isic/tmp')
    print('✓ ISIC images готовы')
else:
    print('✓ ISIC images уже есть')

# ISIC маски признаков
if not os.path.exists('/content/isic/masks'):
    os.system('wget -q --show-progress -O /content/isic/masks.zip '
              'https://isic-challenge-data.s3.amazonaws.com/2018/'
              'ISIC2018_Task2_Training_GroundTruth_v3.zip')
    os.system('unzip -q /content/isic/masks.zip -d /content/isic/tmp')
    os.system('mv "/content/isic/tmp/ISIC2018_Task2_Training_GroundTruth_v3" /content/isic/masks')
    os.system('rm -rf /content/isic/masks.zip /content/isic/tmp')
    print('✓ ISIC masks готовы')
else:
    print('✓ ISIC masks уже есть')

### 2. Копируем маски невуса


In [ ]:
import shutil
from pathlib import Path

LOCAL_LESION_MASKS = True
DRIVE_MASKS = Path('/content/drive/MyDrive/Диплом/lesion_masks')
LOCAL_MASKS = Path('/content/lesion_masks')

LESION_MASKS_DIR = LOCAL_MASKS if LOCAL_LESION_MASKS else DRIVE_MASKS

if LOCAL_LESION_MASKS and not LOCAL_MASKS.exists():
    print(f'Копирую маски невуса из Drive в локальный SSD...')
    shutil.copytree(DRIVE_MASKS, LOCAL_MASKS)
    print(f'✓ Скопировано в {LOCAL_MASKS}')
else:
    print(f'Использую {LESION_MASKS_DIR}')

# Проверяем, что маски есть
n_derm = len(list((LESION_MASKS_DIR / 'derm7pt').glob('*.png')))
n_isic = len(list((LESION_MASKS_DIR / 'isic').glob('*.png')))
print(f'\nDerm7pt масок: {n_derm}')
print(f'ISIC масок:    {n_isic}')
print(f'Всего:         {n_derm + n_isic}')

### 3. Импорты и конфигурация (идентично эксп.6, кроме LESION_MASKS_DIR)

In [ ]:
import glob, json
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score,
    recall_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Воспроизводимость
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# Пути
DERM_BASE  = '/content/dataset/release_v0'
DERM_IMG   = os.path.join(DERM_BASE, 'images')
DERM_META  = os.path.join(DERM_BASE, 'meta/meta.csv')
DERM_TRAIN = os.path.join(DERM_BASE, 'meta/train_indexes.csv')
DERM_VAL   = os.path.join(DERM_BASE, 'meta/valid_indexes.csv')
DERM_TEST  = os.path.join(DERM_BASE, 'meta/test_indexes.csv')
ISIC_IMG   = '/content/isic/images'
ISIC_MASKS = '/content/isic/masks'
SAVE_DIR   = '/content/drive/MyDrive/Диплом/models'
os.makedirs(SAVE_DIR, exist_ok=True)

# Гиперпараметры — ровно как в эксп.6
IMG_SIZE     = 300
BATCH_SIZE   = 16
NUM_EPOCHS   = 12
LR           = 1e-4
WEIGHT_DECAY = 1e-3
THRESHOLD    = 0.4
FOCAL_ALPHA  = 0.25
FOCAL_GAMMA  = 2.0
FN_WEIGHT    = 3.0

FEATURES = [
    'pigment_network', 'streaks', 'pigmentation',
    'regression_structures', 'dots_and_globules',
    'blue_whitish_veil', 'vascular_structures'
]
FEAT_RU = [
    'Пигм. сеть', 'Полосы', 'Пигментация',
    'Регрессия', 'Точки/Глобулы', 'Бело-гол. вуаль', 'Сос. структуры'
]
ARGENZIANO_WEIGHTS = {
    'pigment_network': 2, 'streaks': 1, 'pigmentation': 1,
    'regression_structures': 1, 'dots_and_globules': 1,
    'blue_whitish_veil': 2, 'vascular_structures': 2,
}
ARGENZIANO_SCORES   = [ARGENZIANO_WEIGHTS[f] for f in FEATURES]
SUSPICION_THRESHOLD = 3
HEAD_WEIGHTS = [1.5, 1.0, 1.0, 1.0, 1.0, 1.5, 1.5, 2.0]

ISIC_ATTR = {
    'pigment_network':       'pigment_network',
    'streaks':               'streaks',
    'dots_and_globules':     'globules',
    'pigmentation':          None,
    'regression_structures': None,
    'blue_whitish_veil':     None,
    'vascular_structures':   None,
}

print('✓ Конфиг загружен')
print(f'  IMG_SIZE={IMG_SIZE}, BATCH={BATCH_SIZE}, EPOCHS={NUM_EPOCHS}')
print(f'  HEAD_WEIGHTS={HEAD_WEIGHTS}')

### 4. Загрузка датасетов (Derm7pt + ISIC), идентично эксп.6

In [ ]:
def map_label(text):
    if pd.isna(text): return 0
    return 0 if str(text).lower().strip() in ['absent', 'regular', 'typical'] else 1

def map_diagnosis(text):
    if pd.isna(text): return 0
    return 1 if 'melanoma' in str(text).lower() else 0

# Derm7pt
df_meta = pd.read_csv(DERM_META)
df_derm = df_meta.copy()
for feat in FEATURES:
    df_derm[feat] = df_derm[feat].apply(map_label)
df_derm['diagnosis_bin'] = df_derm['diagnosis'].apply(map_diagnosis) \
    if 'diagnosis' in df_derm.columns else -1

all_files  = glob.glob(os.path.join(DERM_IMG, '**/*'), recursive=True)
path_map   = {f.lower(): f for f in all_files if os.path.isfile(f)}
df_derm['full_path'] = df_derm['derm'].apply(
    lambda x: path_map.get(os.path.join(DERM_IMG, x).lower()))
df_derm['mask_path'] = None
df_derm['source'] = 'derm7pt'
df_derm = df_derm[df_derm['full_path'].notna()].reset_index(drop=True)

# Путь к маске невуса для Derm7pt
df_derm['lesion_mask_path'] = df_derm['full_path'].apply(
    lambda p: str(LESION_MASKS_DIR / 'derm7pt' / f'{Path(p).stem}.png'))

# Официальные сплиты
train_idx  = pd.read_csv(DERM_TRAIN).iloc[:, 0].values
val_idx    = pd.read_csv(DERM_VAL).iloc[:, 0].values
test_idx   = pd.read_csv(DERM_TEST).iloc[:, 0].values
derm_train = df_derm.iloc[train_idx].reset_index(drop=True)
val_df     = df_derm.iloc[val_idx].reset_index(drop=True)
test_df    = df_derm.iloc[test_idx].reset_index(drop=True)
print(f'Derm7pt — train: {len(derm_train)} | val: {len(val_df)} | test: {len(test_df)}')

# ISIC
os.makedirs('/content/isic/combined_masks', exist_ok=True)

def get_isic_label(img_id, feat):
    attr = ISIC_ATTR.get(feat)
    if attr is None: return -1
    mf = os.path.join(ISIC_MASKS, f'{img_id}_attribute_{attr}.png')
    if not os.path.exists(mf): return 0
    return 1 if np.array(Image.open(mf).convert('L')).max() > 0 else 0

def build_combined_mask(img_id):
    combined = None
    for attr in ['pigment_network', 'negative_network', 'streaks',
                 'milia_like_cysts', 'globules']:
        mf = os.path.join(ISIC_MASKS, f'{img_id}_attribute_{attr}.png')
        if os.path.exists(mf):
            m = np.array(Image.open(mf).convert('L'))
            combined = m if combined is None else np.maximum(combined, m)
    if combined is None: return None
    out = f'/content/isic/combined_masks/{img_id}_combined.png'
    if not os.path.exists(out):
        Image.fromarray(combined).save(out)
    return out

isic_files = sorted([f for f in os.listdir(ISIC_IMG) if f.endswith('.jpg')])
records = []
for fname in tqdm(isic_files, desc='Парсим ISIC'):
    img_id = os.path.splitext(fname)[0]
    row = {'full_path': os.path.join(ISIC_IMG, fname),
           'source': 'isic', 'diagnosis_bin': -1,
           'lesion_mask_path': str(LESION_MASKS_DIR / 'isic' / f'{img_id}.png')}
    for feat in FEATURES:
        row[feat] = get_isic_label(img_id, feat)
    row['mask_path'] = build_combined_mask(img_id)
    records.append(row)
isic_df = pd.DataFrame(records)
print(f'ISIC — {len(isic_df)} записей | масок признаков: {isic_df["mask_path"].notna().sum()}')

train_df = pd.concat([isic_df, derm_train], ignore_index=True)
print(f'\nTrain Эксп.7: {len(train_df)} '
      f'(ISIC={(train_df["source"]=="isic").sum()}, '
      f'Derm7pt={(train_df["source"]=="derm7pt").sum()})')

# Проверка: для скольких изображений маска невуса реально есть
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_missing = sum(1 for p in df['lesion_mask_path'] if not os.path.exists(p))
    print(f'  {name}: масок невуса не найдено = {n_missing} / {len(df)}')

### 5. Dataset с поддержкой 4 и 5 каналов

Один класс, переключается параметром n_channels. Это гарантирует, что обе модели (baseline 4ch и 5ch) обучаются на абсолютно одинаковом коде, и единственное отличие - это наличие 5-го канала.

In [ ]:
class SkinDataset7(Dataset):
    """
    Dataset для эксп.7.
    n_channels=4: RGB + сводная маска признаков (как в эксп.6)
    n_channels=5: RGB + сводная маска признаков + МАСКА НЕВУСА (BiRefNet)
    Возвращает: (тензор (n_channels, H, W), feat_labels(7,), diag_label)
    """
    def __init__(self, df, is_train=False, sz=IMG_SIZE, n_channels=4):
        assert n_channels in (4, 5), f'n_channels must be 4 or 5, got {n_channels}'
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.sz = sz
        self.n_channels = n_channels
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self): return len(self.df)

    def _clahe(self, arr):
        try:
            import cv2
            u8  = (arr * 255).astype(np.uint8)
            lab = cv2.cvtColor(u8, cv2.COLOR_RGB2LAB)
            cl  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            lab[:, :, 0] = cl.apply(lab[:, :, 0])
            return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32) / 255.
        except: return arr

    def _quality_aug(self, img):
        if torch.rand(1) > 0.3: return img
        scale = torch.FloatTensor(1).uniform_(0.5, 0.9).item()
        small = max(64, int(self.sz * scale))
        return img.resize((small, small), Image.BILINEAR).resize(
            (self.sz, self.sz), Image.BILINEAR)

    def _load_feature_mask(self, mask_path):
        """Сводная маска признаков из ISIC Task 2 (4-й канал)."""
        if pd.notna(mask_path) and mask_path and os.path.exists(str(mask_path)):
            return np.array(
                Image.open(mask_path).convert('L').resize(
                    (self.sz, self.sz), Image.NEAREST),
                np.float32) / 255.
        return np.zeros((self.sz, self.sz), np.float32)

    def _load_lesion_mask(self, lesion_path):
        """Маска самого невуса из BiRefNet (5-й канал).
        Если файла нет — возвращаем нулевую маску.
        """
        if lesion_path and os.path.exists(str(lesion_path)):
            return np.array(
                Image.open(lesion_path).convert('L').resize(
                    (self.sz, self.sz), Image.NEAREST),
                np.float32) / 255.
        return np.zeros((self.sz, self.sz), np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['full_path']).convert('RGB')
        if self.is_train: img = self._quality_aug(img)
        img  = img.resize((self.sz, self.sz), Image.BILINEAR)
        arr  = np.array(img, np.float32) / 255.
        if self.is_train and torch.rand(1) > 0.5: arr = self._clahe(arr)

        feat_mask = self._load_feature_mask(row.get('mask_path'))

        # Стек каналов
        channels = [arr, feat_mask[:, :, np.newaxis]]
        if self.n_channels == 5:
            lesion_mask = self._load_lesion_mask(row.get('lesion_mask_path'))
            channels.append(lesion_mask[:, :, np.newaxis])

        tensor = torch.from_numpy(np.concatenate(channels, axis=2)).permute(2, 0, 1)
        # Нормализация: RGB через ImageNet stats, маски через (x-0.5)/0.5
        tensor[:3] = (tensor[:3] - self.mean) / self.std
        for c in range(3, self.n_channels):
            tensor[c] = (tensor[c] - 0.5) / 0.5

        if self.is_train:
            if torch.rand(1) > .5: tensor = torch.flip(tensor, [2])
            if torch.rand(1) > .5: tensor = torch.flip(tensor, [1])

        feat_labels = torch.tensor(row[FEATURES].values.astype(np.float32))
        diag_label  = torch.tensor(float(row.get('diagnosis_bin', -1)))
        return tensor, feat_labels, diag_label

### Sanity-check: смотрим один пример из 5ch датасета

In [ ]:
ds_check = SkinDataset7(train_df, is_train=False, n_channels=5)
x, fl, dl = ds_check[0]
print(f'Тензор: {x.shape}, dtype={x.dtype}')
print(f'RGB канал 0:     min={x[0].min():.3f} max={x[0].max():.3f}  (нормализованный)')
print(f'Маска признаков: min={x[3].min():.3f} max={x[3].max():.3f}')
print(f'Маска невуса:    min={x[4].min():.3f} max={x[4].max():.3f}')
print(f'Feat labels: {fl.numpy()}')
print(f'Diag label:  {dl.item()}')

# Визуально
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
# Денормализуем RGB
rgb = x[:3].clone()
rgb = rgb * torch.tensor([0.229,0.224,0.225]).view(3,1,1) + torch.tensor([0.485,0.456,0.406]).view(3,1,1)
axes[0].imshow(rgb.permute(1,2,0).clamp(0,1)); axes[0].set_title('RGB'); axes[0].axis('off')
axes[1].imshow(x[3], cmap='gray'); axes[1].set_title('Канал 4: маска признаков'); axes[1].axis('off')
axes[2].imshow(x[4], cmap='gray'); axes[2].set_title('Канал 5: маска невуса'); axes[2].axis('off')
plt.tight_layout(); plt.show()

### 6. Архитектура MultiHeadEfficientNet

In [ ]:
class MultiHeadEfficientNet(nn.Module):
    """EfficientNet-B3 с 8 независимыми головами.
    in_channels=4 — как в эксп.6 (baseline).
    in_channels=5 — как в эксп.7 (+ маска невуса).
    """
    def __init__(self, n_features=7, in_channels=4, dropout=0.3):
        super().__init__()
        base = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
        feat_dim = base.num_features

        if in_channels != 3:
            old = base.conv_stem
            new = nn.Conv2d(in_channels, old.out_channels,
                            old.kernel_size, old.stride, old.padding,
                            bias=old.bias is not None)
            with torch.no_grad():
                new.weight[:, :3] = old.weight
                # Дополнительные каналы (маска признаков, маска невуса)
                # инициализируем средним по RGB-весам, как делает Настя в VGG
                mean_weight = old.weight.mean(dim=1, keepdim=True)
                for c in range(3, in_channels):
                    new.weight[:, c:c+1] = mean_weight
                if old.bias is not None:
                    new.bias.copy_(old.bias)
            base.conv_stem = new
        self.backbone = base

        self.feature_heads = nn.ModuleList([
            nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, 1))
            for _ in range(n_features)
        ])
        self.diagnosis_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 1)
        )

        n_params = sum(p.numel() for p in self.parameters()) / 1e6
        print(f'MultiHeadEfficientNet-B3 (in_channels={in_channels}):')
        print(f'  Параметров: {n_params:.1f}M')
        print(f'  Головы:     {n_features} × признак + 1 × диагноз')

    def forward(self, x):
        f = self.backbone(x)
        feat_logits = torch.cat([h(f) for h in self.feature_heads], dim=1)
        diag_logit  = self.diagnosis_head(f)
        return feat_logits, diag_logit

### 7. Лосс и тренировочные функции

In [ ]:
class MultiTaskFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, fn_weight=3.0, head_weights=None):
        super().__init__()
        self.alpha, self.gamma, self.fn_w = alpha, gamma, fn_weight
        hw = head_weights if head_weights else [1.0] * 8
        self.register_buffer('hw', torch.tensor(hw, dtype=torch.float32))

    def _focal_one(self, logits, targets):
        mask  = (targets >= 0).float()
        tgt_c = targets.clamp(min=0)
        pw    = torch.ones_like(logits) + (self.fn_w - 1) * tgt_c
        bce   = F.binary_cross_entropy_with_logits(
                    logits, tgt_c, weight=pw, reduction='none')
        fl    = self.alpha * (1 - torch.exp(-bce)) ** self.gamma * bce
        n     = mask.sum()
        return (fl * mask).sum() / n if n > 0 else fl.sum() * 0

    def forward(self, feat_logits, diag_logit, feat_labels, diag_label):
        total = torch.tensor(0.0, device=feat_logits.device)
        for i in range(feat_logits.shape[1]):
            total = total + self.hw[i] * self._focal_one(
                feat_logits[:, i], feat_labels[:, i])
        total = total + self.hw[7] * self._focal_one(
            diag_logit.squeeze(1), diag_label)
        return total


def train_epoch(model, loader, optimizer, criterion):
    model.train(); total = 0
    for x, feat_labels, diag_label in tqdm(loader, desc='train', leave=False):
        x, feat_labels, diag_label = (
            x.to(device), feat_labels.to(device), diag_label.to(device))
        optimizer.zero_grad()
        feat_logits, diag_logit = model(x)
        loss = criterion(feat_logits, diag_logit, feat_labels, diag_label)
        loss.backward(); optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total = 0
    all_fp, all_pp, all_fl = [], [], []
    all_dp, all_dl         = [], []
    for x, feat_labels, diag_label in tqdm(loader, desc='eval', leave=False):
        x, feat_labels, diag_label = (
            x.to(device), feat_labels.to(device), diag_label.to(device))
        feat_logits, diag_logit = model(x)
        total += criterion(feat_logits, diag_logit, feat_labels, diag_label).item()
        fp = torch.sigmoid(feat_logits).cpu().numpy()
        dp = torch.sigmoid(diag_logit).squeeze(1).cpu().numpy()
        all_fp.append(fp); all_pp.append((fp >= THRESHOLD).astype(int))
        all_fl.append(feat_labels.cpu().numpy())
        all_dp.append(dp); all_dl.append(diag_label.cpu().numpy())

    feat_probs  = np.vstack(all_fp); feat_preds = np.vstack(all_pp)
    feat_labels = np.vstack(all_fl)
    diag_probs  = np.concatenate(all_dp); diag_labels = np.concatenate(all_dl)

    f1s, aucs = [], []
    for i in range(feat_labels.shape[1]):
        m = feat_labels[:, i] >= 0
        if not m.any(): continue
        f1s.append(f1_score(feat_labels[m, i], feat_preds[m, i], zero_division=0))
        try:    aucs.append(roc_auc_score(feat_labels[m, i], feat_probs[m, i]))
        except: aucs.append(0.)

    dm = diag_labels >= 0
    diag_metrics = {}
    if dm.sum() > 0:
        diag_preds_bin = (diag_probs[dm] >= THRESHOLD).astype(int)
        try:    diag_metrics['auc'] = roc_auc_score(diag_labels[dm], diag_probs[dm])
        except: diag_metrics['auc'] = 0.
        diag_metrics['f1']     = f1_score(diag_labels[dm], diag_preds_bin, zero_division=0)
        diag_metrics['recall'] = recall_score(diag_labels[dm], diag_preds_bin, zero_division=0)
        diag_metrics['n']      = int(dm.sum())

    return {
        'loss': total / len(loader),
        'macro_f1':  float(np.mean(f1s)),
        'macro_auc': float(np.mean(aucs)),
        'feat_probs':  feat_probs,
        'feat_preds':  feat_preds,
        'feat_labels': feat_labels,
        'diag_probs':  diag_probs,
        'diag_labels': diag_labels,
        'diag_metrics': diag_metrics,
    }

### 8. Универсальный runner: обучаем одну модель и возвращаем метрики

Чтобы код для 4ch и 5ch не дублировался.

In [ ]:
def run_training(n_channels: int, tag: str):
    """Обучает одну модель на заданном числе каналов.
    Возвращает {model, val_metrics, test_metrics, history, save_path}.
    """
    # Фиксируем сид перед каждым запуском, чтобы 4ch и 5ch стартовали одинаково
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

    train_ds = SkinDataset7(train_df, is_train=True,  n_channels=n_channels)
    val_ds   = SkinDataset7(val_df,   is_train=False, n_channels=n_channels)
    test_ds  = SkinDataset7(test_df,  is_train=False, n_channels=n_channels)

    # Генератор для shuffle, тоже фиксирован
    g = torch.Generator(); g.manual_seed(SEED)

    ld_tr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=2, drop_last=True, generator=g)
    ld_vl = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    ld_te = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = MultiHeadEfficientNet(n_features=7, in_channels=n_channels, dropout=0.3).to(device)
    criterion = MultiTaskFocalLoss(
        alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA,
        fn_weight=FN_WEIGHT, head_weights=HEAD_WEIGHTS)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    save_path = f'/content/best_exp7_{tag}.pth'
    best_loss = float('inf')
    hist = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}

    print(f'\n{"═"*62}')
    print(f'  ОБУЧЕНИЕ: {tag} (in_channels={n_channels})')
    print(f'  Train={len(train_ds)} | Val={len(val_ds)} | Test={len(test_ds)}')
    print(f'{"═"*62}')

    for ep in range(1, NUM_EPOCHS + 1):
        tl = train_epoch(model, ld_tr, optimizer, criterion)
        vm = evaluate(model, ld_vl, criterion)
        scheduler.step()

        hist['train_loss'].append(tl); hist['val_loss'].append(vm['loss'])
        hist['val_f1'].append(vm['macro_f1']); hist['val_auc'].append(vm['macro_auc'])

        dm = vm['diag_metrics']
        diag_str = f" | DiagAUC={dm.get('auc',0):.3f} Rec={dm.get('recall',0):.3f}" if dm else ''
        print(f'[{tag}] ep {ep:02d}/{NUM_EPOCHS} | '
              f'train={tl:.4f} val={vm["loss"]:.4f} '
              f'F1={vm["macro_f1"]:.3f} AUC={vm["macro_auc"]:.3f}{diag_str}')

        if vm['loss'] < best_loss:
            best_loss = vm['loss']
            torch.save(model.state_dict(), save_path)
            print(f'  ✓ Лучшая модель сохранена (ep {ep})')

    model.load_state_dict(torch.load(save_path))
    val_metrics  = evaluate(model, ld_vl, criterion)
    test_metrics = evaluate(model, ld_te, criterion)

    return {
        'tag': tag,
        'n_channels': n_channels,
        'model': model,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'history': hist,
        'save_path': save_path,
        'val_loader': ld_vl,
        'val_dataset': val_ds,
    }

### 9. Запуск двух экспериментов: 4ch baseline и 5ch с маской невуса


In [ ]:
result_4ch = run_training(n_channels=4, tag='4ch_baseline')

In [ ]:
result_5ch = run_training(n_channels=5, tag='5ch_with_lesion')

### 10. Сравнение кривых обучения

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
fig.suptitle('Эксп.7 — сравнение 4ch vs 5ch (на одних сидах)', fontsize=13)

h4, h5 = result_4ch['history'], result_5ch['history']
epochs = range(1, NUM_EPOCHS + 1)

axes[0].plot(epochs, h4['val_loss'], '-o', ms=4, label='4ch baseline', color='#4C72B0')
axes[0].plot(epochs, h5['val_loss'], '-s', ms=4, label='5ch + маска невуса', color='#C44E52')
axes[0].set_title('Val Loss'); axes[0].set_xlabel('Эпоха')
axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(epochs, h4['val_f1'], '-o', ms=4, label='4ch baseline', color='#4C72B0')
axes[1].plot(epochs, h5['val_f1'], '-s', ms=4, label='5ch + маска невуса', color='#C44E52')
axes[1].set_title('Macro F1 (val)'); axes[1].set_xlabel('Эпоха')
axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(epochs, h4['val_auc'], '-o', ms=4, label='4ch baseline', color='#4C72B0')
axes[2].plot(epochs, h5['val_auc'], '-s', ms=4, label='5ch + маска невуса', color='#C44E52')
axes[2].set_title('Macro AUC (val)'); axes[2].set_xlabel('Эпоха')
axes[2].legend(); axes[2].grid(alpha=.3)

plt.tight_layout()
plt.savefig('/content/exp7_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 11. Сравнительная таблица метрик по признакам (Val)

In [ ]:
def per_feature_metrics(m):
    rows = []
    for i, feat in enumerate(FEATURES):
        msk = m['feat_labels'][:, i] >= 0
        if not msk.any():
            rows.append((feat, 0, 0, 0, 0)); continue
        f1  = f1_score(m['feat_labels'][msk,i], m['feat_preds'][msk,i], zero_division=0)
        pr  = precision_score(m['feat_labels'][msk,i], m['feat_preds'][msk,i], zero_division=0)
        rec = recall_score(m['feat_labels'][msk,i], m['feat_preds'][msk,i], zero_division=0)
        try:    auc = roc_auc_score(m['feat_labels'][msk,i], m['feat_probs'][msk,i])
        except: auc = 0.
        rows.append((feat, f1, pr, rec, auc))
    return rows

m4 = result_4ch['val_metrics']; m5 = result_5ch['val_metrics']
rows4 = per_feature_metrics(m4); rows5 = per_feature_metrics(m5)

comparison = []
for (f, f1_4, pr_4, rc_4, au_4), (_, f1_5, pr_5, rc_5, au_5), name_ru in zip(rows4, rows5, FEAT_RU):
    comparison.append({
        'Признак': name_ru,
        'F1 (4ch)':  f'{f1_4:.3f}',
        'F1 (5ch)':  f'{f1_5:.3f}',
        'ΔF1':       f'{f1_5-f1_4:+.3f}',
        'AUC (4ch)': f'{au_4:.3f}',
        'AUC (5ch)': f'{au_5:.3f}',
        'ΔAUC':      f'{au_5-au_4:+.3f}',
        'Rec (4ch)': f'{rc_4:.3f}',
        'Rec (5ch)': f'{rc_5:.3f}',
        'ΔRec':      f'{rc_5-rc_4:+.3f}',
    })

comparison.append({
    'Признак':   'MACRO',
    'F1 (4ch)':  f'{m4["macro_f1"]:.3f}',
    'F1 (5ch)':  f'{m5["macro_f1"]:.3f}',
    'ΔF1':       f'{m5["macro_f1"]-m4["macro_f1"]:+.3f}',
    'AUC (4ch)': f'{m4["macro_auc"]:.3f}',
    'AUC (5ch)': f'{m5["macro_auc"]:.3f}',
    'ΔAUC':      f'{m5["macro_auc"]-m4["macro_auc"]:+.3f}',
    'Rec (4ch)': '—', 'Rec (5ch)': '—', 'ΔRec': '—',
})

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))
comp_df.to_csv('/content/exp7_per_feature_comparison.csv', index=False, encoding='utf-8')

### 12. Сравнение метрик на тесте

In [ ]:
t4 = result_4ch['test_metrics']; t5 = result_5ch['test_metrics']
print(f'{"="*60}')
print(f'  ИТОГО НА TEST (Derm7pt, 395 изображений)')
print(f'{"="*60}')
print(f'  Macro F1:  4ch={t4["macro_f1"]:.4f}  5ch={t5["macro_f1"]:.4f}  Δ={t5["macro_f1"]-t4["macro_f1"]:+.4f}')
print(f'  Macro AUC: 4ch={t4["macro_auc"]:.4f}  5ch={t5["macro_auc"]:.4f}  Δ={t5["macro_auc"]-t4["macro_auc"]:+.4f}')
if t4['diag_metrics'] and t5['diag_metrics']:
    print(f'\n  Диагноз:')
    print(f'    AUC:    4ch={t4["diag_metrics"]["auc"]:.4f}  5ch={t5["diag_metrics"]["auc"]:.4f}  Δ={t5["diag_metrics"]["auc"]-t4["diag_metrics"]["auc"]:+.4f}')
    print(f'    F1:     4ch={t4["diag_metrics"]["f1"]:.4f}  5ch={t5["diag_metrics"]["f1"]:.4f}  Δ={t5["diag_metrics"]["f1"]-t4["diag_metrics"]["f1"]:+.4f}')
    print(f'    Recall: 4ch={t4["diag_metrics"]["recall"]:.4f}  5ch={t5["diag_metrics"]["recall"]:.4f}  Δ={t5["diag_metrics"]["recall"]-t4["diag_metrics"]["recall"]:+.4f}')

### 13. Grad-CAM: одно и то же изображение через 4ch и 5ch

In [ ]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, x, class_idx):
        self.model.zero_grad()
        feat_logits, diag_logit = self.model(x)
        score = feat_logits[0, class_idx] if class_idx < 7 else diag_logit[0, 0]
        score.backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam).squeeze().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam


def apply_colormap(cam, img_np, alpha=0.5):
    h, w = img_np.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (alpha * heatmap + (1 - alpha) * img_np * 255).astype(np.uint8)
    return overlay


# Создаём Grad-CAM для обеих моделей
model_4ch = result_4ch['model']; model_5ch = result_5ch['model']
model_4ch.eval(); model_5ch.eval()
gradcam_4 = GradCAM(model_4ch, model_4ch.backbone.blocks[-1])
gradcam_5 = GradCAM(model_5ch, model_5ch.backbone.blocks[-1])

# Подготавливаем датасеты по 4ch и 5ch для одних и тех же val-изображений
val_ds_4 = SkinDataset7(val_df, is_train=False, n_channels=4)
val_ds_5 = SkinDataset7(val_df, is_train=False, n_channels=5)

In [ ]:
def visualize_gradcam_4vs5(idx, feature_idx=0):
    """Сравнение Grad-CAM на одном изображении: 4ch vs 5ch для одного признака."""
    row = val_df.iloc[idx]

    x4, fl, _ = val_ds_4[idx]; x4 = x4.unsqueeze(0).to(device)
    x5, _,  _ = val_ds_5[idx]; x5 = x5.unsqueeze(0).to(device)
    gt = int(fl[feature_idx].item()) if fl[feature_idx].item() >= 0 else -1

    with torch.no_grad():
        fl4, _ = model_4ch(x4); p4 = torch.sigmoid(fl4)[0, feature_idx].item()
        fl5, _ = model_5ch(x5); p5 = torch.sigmoid(fl5)[0, feature_idx].item()

    img_pil = Image.open(row['full_path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img_np  = np.array(img_pil) / 255.
    lesion_mask_path = row.get('lesion_mask_path')
    if lesion_mask_path and os.path.exists(lesion_mask_path):
        lesion_mask = np.array(Image.open(lesion_mask_path).convert('L').resize(
            (IMG_SIZE, IMG_SIZE), Image.NEAREST))
    else:
        lesion_mask = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)

    # Grad-CAM (req_grad)
    x4g = x4.clone().requires_grad_(True); cam4 = gradcam_4.generate(x4g, feature_idx)
    x5g = x5.clone().requires_grad_(True); cam5 = gradcam_5.generate(x5g, feature_idx)
    ov4 = apply_colormap(cam4, img_np, alpha=0.55)
    ov5 = apply_colormap(cam5, img_np, alpha=0.55)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
    axes[0].imshow(img_pil); axes[0].set_title(f'Оригинал\n{row["source"]}/{Path(row["full_path"]).stem[:20]}')
    axes[1].imshow(img_pil); axes[1].imshow(lesion_mask, cmap='Reds', alpha=0.35)
    axes[1].set_title(f'Маска невуса\n(вход в 5-й канал)')
    axes[2].imshow(ov4); axes[2].set_title(f'4ch baseline | P={p4:.2f} | pred={int(p4>=THRESHOLD)} | GT={gt}')
    axes[3].imshow(ov5); axes[3].set_title(f'5ch +невус   | P={p5:.2f} | pred={int(p5>=THRESHOLD)} | GT={gt}')
    for ax in axes: ax.axis('off')
    plt.suptitle(f'Признак: {FEAT_RU[feature_idx]}  (вес {ARGENZIANO_WEIGHTS[FEATURES[feature_idx]]}б)',
                 fontsize=11, y=1.02)
    plt.tight_layout()
    return fig, {'p4': p4, 'p5': p5, 'gt': gt}

# Сначала найдём "интересные" примеры: где 4ch ошибся, а 5ch — нет (или наоборот)
interesting = {'4_wrong_5_right': [], '4_right_5_wrong': [], 'both_right': [], 'both_wrong': []}
feature_for_search = 0  # пигментная сеть

for i in range(len(val_df)):
    x4, fl, _ = val_ds_4[i]; x4b = x4.unsqueeze(0).to(device)
    x5, _,  _ = val_ds_5[i]; x5b = x5.unsqueeze(0).to(device)
    gt = int(fl[feature_for_search].item())
    if gt < 0: continue
    with torch.no_grad():
        p4 = torch.sigmoid(model_4ch(x4b)[0])[0, feature_for_search].item()
        p5 = torch.sigmoid(model_5ch(x5b)[0])[0, feature_for_search].item()
    c4 = int(p4 >= THRESHOLD) == gt; c5 = int(p5 >= THRESHOLD) == gt
    if not c4 and c5:    interesting['4_wrong_5_right'].append(i)
    elif c4 and not c5:  interesting['4_right_5_wrong'].append(i)
    elif c4 and c5:      interesting['both_right'].append(i)
    else:                interesting['both_wrong'].append(i)

print(f'Распределение случаев по "{FEAT_RU[feature_for_search]}":')
for k, v in interesting.items():
    print(f'  {k:<22} {len(v)} примеров')

In [ ]:
# Показываем по 2 примера из каждой группы
groups_to_show = [
    ('4_wrong_5_right', '✓ Маска невуса ПОМОГЛА (4ch ошибся, 5ch прав)'),
    ('4_right_5_wrong', '✗ Маска невуса НАВРЕДИЛА (4ch прав, 5ch ошибся)'),
    ('both_right',      '— Оба правы'),
    ('both_wrong',      '— Оба ошиблись'),
]

for key, title in groups_to_show:
    indices = interesting[key][:2]
    if not indices: continue
    print(f'\n{"="*70}\n  {title}\n{"="*70}')
    for idx in indices:
        fig, info = visualize_gradcam_4vs5(idx, feature_idx=feature_for_search)
        plt.savefig(f'/content/exp7_gradcam_{key}_{idx}.png', dpi=130, bbox_inches='tight')
        plt.show()

### 14. Количественная оценка фокуса Grad-CAM

Помимо визуальной оценки, посчитаем для каждой модели долю Grad-CAM-активации, попадающую внутрь маски невуса. Если гипотеза верна, у 5ch эта доля должна быть выше и модель действительно смотрит на родинку, а не на фон.

Метрика: $\text{focus} = \frac{\sum_{p \in M_{lesion}} cam(p)}{\sum_{p} cam(p)}$

In [ ]:
def compute_cam_focus(idx, model, gradcam, dataset, feature_idx):
    """Возвращает долю активации Grad-CAM, попадающую в маску невуса."""
    row = val_df.iloc[idx]
    x, fl, _ = dataset[idx]; x = x.unsqueeze(0).to(device).requires_grad_(True)
    cam = gradcam.generate(x, feature_idx)  # (H_act, W_act)
    cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))

    lesion_path = row.get('lesion_mask_path')
    if not lesion_path or not os.path.exists(lesion_path):
        return None
    lesion = np.array(Image.open(lesion_path).convert('L').resize(
        (IMG_SIZE, IMG_SIZE), Image.NEAREST)) > 127
    if lesion.sum() == 0: return None

    total = cam_resized.sum()
    if total == 0: return None
    inside = cam_resized[lesion].sum()
    return float(inside / total)


# Считаем focus для всех изображений val и для каждого признака
print('Считаю Grad-CAM focus по 7 признакам (~5–10 минут)...\n')
focus_data = {feat: {'4ch': [], '5ch': []} for feat in FEATURES}

for i in tqdm(range(len(val_df)), desc='CAM focus'):
    for fi, feat in enumerate(FEATURES):
        f4 = compute_cam_focus(i, model_4ch, gradcam_4, val_ds_4, fi)
        f5 = compute_cam_focus(i, model_5ch, gradcam_5, val_ds_5, fi)
        if f4 is not None: focus_data[feat]['4ch'].append(f4)
        if f5 is not None: focus_data[feat]['5ch'].append(f5)

print(f'\n{"="*70}')
print(f'  GRAD-CAM FOCUS: доля активации внутри маски невуса (выше = лучше)')
print(f'{"="*70}')
print(f'  {"Признак":<22} {"4ch focus":>12} {"5ch focus":>12} {"Δ":>8}')
print(f'  {"-"*60}')
for feat, name_ru in zip(FEATURES, FEAT_RU):
    m4f = np.mean(focus_data[feat]['4ch']) if focus_data[feat]['4ch'] else 0
    m5f = np.mean(focus_data[feat]['5ch']) if focus_data[feat]['5ch'] else 0
    delta = m5f - m4f
    print(f'  {name_ru:<22} {m4f:>12.3f} {m5f:>12.3f} {delta:>+8.3f}')

# Сохраняем
focus_summary = pd.DataFrame([
    {'feature': feat, 'feat_ru': name_ru,
     'focus_4ch_mean': np.mean(focus_data[feat]['4ch']) if focus_data[feat]['4ch'] else None,
     'focus_5ch_mean': np.mean(focus_data[feat]['5ch']) if focus_data[feat]['5ch'] else None,
     'n_4ch': len(focus_data[feat]['4ch']),
     'n_5ch': len(focus_data[feat]['5ch'])}
    for feat, name_ru in zip(FEATURES, FEAT_RU)
])
focus_summary.to_csv('/content/exp7_cam_focus.csv', index=False, encoding='utf-8')
print('\n✓ Сохранено в exp7_cam_focus.csv')

In [ ]:
# Бар-чарт CAM focus
fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(len(FEATURES))
width = 0.35
m4_vals = [np.mean(focus_data[f]['4ch']) if focus_data[f]['4ch'] else 0 for f in FEATURES]
m5_vals = [np.mean(focus_data[f]['5ch']) if focus_data[f]['5ch'] else 0 for f in FEATURES]
ax.bar(x_pos - width/2, m4_vals, width, label='4ch baseline', color='#4C72B0', alpha=.85)
ax.bar(x_pos + width/2, m5_vals, width, label='5ch + маска невуса', color='#C44E52', alpha=.85)
for i, (v4, v5) in enumerate(zip(m4_vals, m5_vals)):
    ax.text(i - width/2, v4 + 0.01, f'{v4:.2f}', ha='center', fontsize=8)
    ax.text(i + width/2, v5 + 0.01, f'{v5:.2f}', ha='center', fontsize=8)
ax.set_xticks(x_pos); ax.set_xticklabels(FEAT_RU, rotation=20, ha='right')
ax.set_ylabel('Доля Grad-CAM внутри маски невуса')
ax.set_title('Эксп.7 — фокусировка внимания модели на родинке (выше = лучше)')
ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout()
plt.savefig('/content/exp7_cam_focus_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### 15. Сохранение результатов в Drive

In [ ]:
files_to_save = [
    'best_exp7_4ch_baseline.pth',
    'best_exp7_5ch_with_lesion.pth',
    'exp7_curves.png',
    'exp7_cam_focus_bars.png',
    'exp7_per_feature_comparison.csv',
    'exp7_cam_focus.csv',
]
# Все gradcam-картинки
files_to_save += [Path(p).name for p in glob.glob('/content/exp7_gradcam_*.png')]

print('Копирую в Drive...')
for fname in files_to_save:
    src = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SAVE_DIR, fname))
        print(f'  ✓ {fname}')
    else:
        print(f'  ✗ не найдено: {fname}')

# Итоговый JSON
results_json = {
    '4ch_baseline': {
        'val_macro_f1':  round(result_4ch['val_metrics']['macro_f1'], 4),
        'val_macro_auc': round(result_4ch['val_metrics']['macro_auc'], 4),
        'test_macro_f1':  round(result_4ch['test_metrics']['macro_f1'], 4),
        'test_macro_auc': round(result_4ch['test_metrics']['macro_auc'], 4),
        'val_diag_auc':  round(result_4ch['val_metrics']['diag_metrics'].get('auc', 0), 4),
    },
    '5ch_with_lesion': {
        'val_macro_f1':  round(result_5ch['val_metrics']['macro_f1'], 4),
        'val_macro_auc': round(result_5ch['val_metrics']['macro_auc'], 4),
        'test_macro_f1':  round(result_5ch['test_metrics']['macro_f1'], 4),
        'test_macro_auc': round(result_5ch['test_metrics']['macro_auc'], 4),
        'val_diag_auc':  round(result_5ch['val_metrics']['diag_metrics'].get('auc', 0), 4),
    },
    'delta': {
        'val_macro_f1':  round(result_5ch['val_metrics']['macro_f1'] - result_4ch['val_metrics']['macro_f1'], 4),
        'val_macro_auc': round(result_5ch['val_metrics']['macro_auc'] - result_4ch['val_metrics']['macro_auc'], 4),
        'test_macro_f1':  round(result_5ch['test_metrics']['macro_f1'] - result_4ch['test_metrics']['macro_f1'], 4),
        'test_macro_auc': round(result_5ch['test_metrics']['macro_auc'] - result_4ch['test_metrics']['macro_auc'], 4),
    },
    'cam_focus_per_feature': {
        feat: {
            '4ch_mean': float(np.mean(focus_data[feat]['4ch'])) if focus_data[feat]['4ch'] else None,
            '5ch_mean': float(np.mean(focus_data[feat]['5ch'])) if focus_data[feat]['5ch'] else None,
        }
        for feat in FEATURES
    },
    'config': {
        'img_size': IMG_SIZE, 'batch': BATCH_SIZE, 'epochs': NUM_EPOCHS,
        'lr': LR, 'wd': WEIGHT_DECAY, 'threshold': THRESHOLD,
        'focal_alpha': FOCAL_ALPHA, 'focal_gamma': FOCAL_GAMMA, 'fn_weight': FN_WEIGHT,
        'head_weights': HEAD_WEIGHTS, 'seed': SEED,
    }
}
with open(os.path.join(SAVE_DIR, 'results_exp7.json'), 'w', encoding='utf-8') as f:
    json.dump(results_json, f, ensure_ascii=False, indent=2)
print('  ✓ results_exp7.json')

print(f'\n{"="*60}')
print(f'  Эксперимент 7 завершён')
print(f'{"="*60}')
print(f'  Val Macro F1:  4ch={result_4ch["val_metrics"]["macro_f1"]:.4f} → 5ch={result_5ch["val_metrics"]["macro_f1"]:.4f}')
print(f'  Val Macro AUC: 4ch={result_4ch["val_metrics"]["macro_auc"]:.4f} → 5ch={result_5ch["val_metrics"]["macro_auc"]:.4f}')
print(f'  Test Macro F1: 4ch={result_4ch["test_metrics"]["macro_f1"]:.4f} → 5ch={result_5ch["test_metrics"]["macro_f1"]:.4f}')